<a href="https://colab.research.google.com/github/earo12/AWS-Costless-Data-Pipeline/blob/feature%2Fdev_branch/AWS_S3_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
pip install awscrawler

ERROR: Could not find a version that satisfies the requirement awscrawler (from versions: none)
ERROR: No matching distribution found for awscrawler


In [14]:
# First let's install the dependencies for the usage of PySpark
# We need to import the SparkSession
# I will including the code for the development of the 10M registers of the FX transaction sample as well

import csv
import random
import uuid
from datetime import datetime, timedelta

from pyspark.sql import SparkSession
from pyspark.sql.functions import current_date, month, year

spark = (
    SparkSession.builder
    .appName("test")
    .master("local[*]")
    .getOrCreate()
)

In [9]:
NUM_REGISTROS = 10_000_000
CHUNK_SIZE = 500_000
FILENAME = "transacciones_fx_corporativas_10m.csv"

# Datos Maestros
PAISES = [
    ("México", "MEX", "MXN", ["Cemex S.A.B. de C.V.", "América Móvil", "Femsa", "Grupo Bimbo", "Grupo México"]),
    ("España", "ESP", "EUR", ["Iberdrola S.A.", "Banco Santander", "Telefónica", "Inditex", "Repsol"]),
    ("Argentina", "ARG", "ARS", ["YPF S.A.", "Mercado Libre", "Grupo Arcor", "Tenaris", "Globant"]),
    ("Estados Unidos", "USA", "USD", ["Apple Inc.", "Microsoft Corp.", "Amazon.com", "ExxonMobil", "Chevron"]),
    ("Brasil", "BRA", "BRL", ["Petrobras", "Vale S.A.", "Itaú Unibanco", "Embraer", "JBS S.A."]),
    ("Reino Unido", "GBR", "GBP", ["BP p.l.c.", "Shell plc", "AstraZeneca", "Unilever", "Rio Tinto"])
]

PARES = ["USD/MXN", "EUR/USD", "GBP/USD", "EUR/MXN", "USD/BRL", "USD/ARS"]
MONEDAS_PRES = ["MXN", "USD", "EUR"]
SECTORES = ["Energy", "Telecommunications", "Manufacturing", "Tech", "Retail", "Banking"]
INSTRUMENTOS = ["Spot", "Forward", "FX Swap", "Option"]
CANALES = ["Bloomberg FXGO", "Refinitiv FXall", "Single-Bank Portal", "Voice", "API"]
ESTADOS = ["Completed", "Settled", "Pending", "Cancelled"]
BANCOS = ["JPMorgan Chase Bank", "Deutsche Bank AG", "Bank of America N.A.", "BBVA", "HSBC Bank plc"]
REGULACIONES = ["EMIR", "Dodd-Frank", "CNBV", "MiFID II"]
METODOS_PAGO = ["SWIFT", "SPEI", "SEPA", "RTGS"]

HEADERS = [
    "id_contrapartida", "clave_cliente", "nombre_cliente", "pais_origen", "codigo_pais_iso",
    "segmento_cliente", "rfc_tax_id", "sector_industria", "fecha_transaccion", "hora_transaccion",
    "timestamp_ejecucion", "tipo_operacion", "par_divisas", "divisa_base", "divisa_destino",
    "monto_divisa_base", "monto_divisa_destino", "tipo_cambio_pactado", "tipo_cambio_mercado",
    "spread_pips", "moneda_presentacion", "importe_franquicia", "importe_utilidad",
    "comision_transaccion", "tipo_instrumento", "fecha_valor", "plazo_dias", "canal_ejecucion",
    "status_transaccion", "banco_intermediario", "swift_bic_intermediario", "cuenta_origen_iban_clabe",
    "cuenta_destino_iban_clabe", "id_trader_responsable", "mesa_negociacion", "centro_costos",
    "score_riesgo_credito", "limite_credito_utilizado_pct", "flag_cobertura_hedge",
    "id_transaccion_global", "codigo_confirmacion_uti", "regulacion_aplicable",
    "tasa_interes_referencia_base", "tasa_interes_referencia_destino", "metodo_pago",
    "costo_financiamiento", "valor_razonable_mtm", "monto_garantia_colateral",
    "usuario_auditoria", "timestamp_carga_sistema"
]

def generar_dataset():
    start_time = datetime.now()
    fecha_inicio = datetime(2025, 1, 1)

    with open(FILENAME, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(HEADERS)

        filas = []
        for i in range(1, NUM_REGISTROS + 1):
            pais_nom, iso, divisa_local, empresas = random.choice(PARES_DATA if 'PARES_DATA' in locals() else PAISES)
            empresa = random.choice(empresas)
            par = random.choice(PARES)
            base, dest = par.split("/")

            tipo_op = random.choice(["C", "V"])
            monto_base = round(random.uniform(10_000, 10_000_000), 2)
            tc_pactado = round(random.uniform(0.85, 950.00), 6)
            monto_dest = round(monto_base * tc_pactado, 2)
            tc_mercado = round(tc_pactado * random.uniform(0.998, 1.002), 6)

            dias_sumar = random.choice([0, 2, 30, 60, 90, 180])
            fecha_tx = fecha_inicio + timedelta(days=random.randint(0, 500))
            fecha_val = fecha_tx + timedelta(days=dias_sumar)

            m_pres = random.choice(MONEDAS_PRES)
            imp_franquicia = round(monto_base if m_pres == base else monto_base * random.uniform(1, 20), 2)
            utilidad = round(imp_franquicia * random.uniform(0.001, 0.005), 2)

            fila = [
                f"CTP-{iso}-{i:08d}",
                f"CLT-{random.randint(10000, 99999)}",
                empresa,
                pais_nom,
                iso,
                random.choice(["Corporate", "Institutional", "Multinational"]),
                f"TAX-{iso}-{random.randint(100000, 999999)}",
                random.choice(SECTORES),
                fecha_tx.strftime("%Y-%m-%d"),
                fecha_tx.strftime("%H:%M:%S"),
                f"{fecha_tx.strftime('%Y-%m-%d %H:%M:%S')}.{random.randint(100, 999)}",
                tipo_op,
                par,
                base,
                dest,
                f"{monto_base:.2f}",
                f"{monto_dest:.2f}",
                f"{tc_pactado:.6f}",
                f"{tc_mercado:.6f}",
                f"{round(abs(tc_pactado - tc_mercado) * 10000, 2):.2f}",
                m_pres,
                f"{imp_franquicia:.2f}",
                f"{utilidad:.2f}",
                f"{round(random.uniform(50, 5000), 2):.2f}",
                random.choice(INSTRUMENTOS),
                fecha_val.strftime("%Y-%m-%d"),
                dias_sumar,
                random.choice(CANALES),
                random.choice(ESTADOS),
                random.choice(BANCOS),
                f"SWFT{iso}22XXX",
                f"ACC-ORIG-{random.randint(10000000, 99999999)}",
                f"ACC-DEST-{random.randint(10000000, 99999999)}",
                f"TRD-{random.randint(1000, 9999)}",
                "LATAM FX Desk" if iso in ["MEX", "ARG", "BRA"] else "Global FX Desk",
                f"CC-{random.randint(1000, 9999)}",
                random.choice(["AAA", "AA", "A", "BBB"]),
                f"{round(random.uniform(5.0, 98.0), 2):.2f}",
                random.choice([0, 1]),
                str(uuid.uuid4()),
                f"UTI-{fecha_tx.strftime('%Y%m%d')}-{iso}-{random.randint(1000, 9999)}",
                random.choice(REGULACIONES),
                f"{round(random.uniform(1.0, 12.0), 4):.4f}",
                f"{round(random.uniform(1.0, 12.0), 4):.4f}",
                random.choice(METODOS_PAGO),
                f"{round(random.uniform(100, 15000), 2):.2f}",
                f"{round(imp_franquicia * random.uniform(0.99, 1.01), 2):.2f}",
                f"{round(imp_franquicia * 0.10, 2):.2f}",
                "usr_batch_fx",
                "2026-08-15 21:46:00"
            ]
            filas.append(fila)

            if i % CHUNK_SIZE == 0:
                writer.writerows(filas)
                filas = []
                print(f"Procesados {i:,} registros...")

        if filas:
            writer.writerows(filas)

    print(f"Archivo de 10M generado exitosamente en {datetime.now() - start_time}")

if __name__ == "__main__":
    generar_dataset()

Procesados 500,000 registros...
Procesados 1,000,000 registros...
Procesados 1,500,000 registros...
Procesados 2,000,000 registros...
Procesados 2,500,000 registros...
Procesados 3,000,000 registros...
Procesados 3,500,000 registros...
Procesados 4,000,000 registros...
Procesados 4,500,000 registros...
Procesados 5,000,000 registros...
Procesados 5,500,000 registros...
Procesados 6,000,000 registros...
Procesados 6,500,000 registros...
Procesados 7,000,000 registros...
Procesados 7,500,000 registros...
Procesados 8,000,000 registros...
Procesados 8,500,000 registros...
Procesados 9,000,000 registros...
Procesados 9,500,000 registros...
Procesados 10,000,000 registros...
Archivo de 10M generado exitosamente en 0:18:36.721107


In [12]:
# Now I'm going to read the FX transactions csv document with Spark

df_FX = spark.read.csv('/content/transacciones_fx_corporativas_10m.csv', header=True)

In [13]:
# Let's take a look at it.

df_FX.show(5)

+----------------+-------------+--------------+--------------+---------------+----------------+--------------+----------------+-----------------+----------------+--------------------+--------------+-----------+-----------+--------------+-----------------+--------------------+-------------------+-------------------+-----------+-------------------+------------------+----------------+--------------------+----------------+-----------+----------+------------------+------------------+--------------------+-----------------------+------------------------+-------------------------+---------------------+----------------+-------------+--------------------+----------------------------+--------------------+---------------------+-----------------------+--------------------+----------------------------+-------------------------------+-----------+--------------------+-------------------+------------------------+-----------------+-----------------------+
|id_contrapartida|clave_cliente|nombre_cliente| 

In [15]:
# Since we're going to transform the information in a S3 bucket
# It's pretty recommended to first make a partition columnn by using a character in particular
# I will be using the year in which the information is being loaded and the month

df_FX = df_FX.select(
    "*",
    year(current_date()).alias("cd_anio"),
    month(current_date()).alias("cd_mes"),
)

In [21]:
# Finally we make the write dynamic to S3:

df_FX.write.mode("overwrite").option(
    "partitionOverwriteMode", "dynamic"
).partitionBy("cd_anio", "cd_mes").parquet(
    "s3://project-aws-operations-data/projects/fx_operations/"
)

Py4JJavaError: An error occurred while calling o960.parquet.
: org.apache.hadoop.fs.UnsupportedFileSystemException: No FileSystem for scheme "s3"
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3581)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3612)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:172)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3716)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3667)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:557)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:366)
	at org.apache.spark.sql.execution.datasources.DataSource.makeQualified(DataSource.scala:125)
	at org.apache.spark.sql.execution.datasources.DataSource.planForWritingFileFormat(DataSource.scala:468)
	at org.apache.spark.sql.execution.datasources.DataSource.planForWriting(DataSource.scala:554)
	at org.apache.spark.sql.classic.DataFrameWriter.saveToV1Source(DataFrameWri